# R-NaD Slay the Spire 2 Offline Training on Colab

This notebook sets up the environment and runs the offline training script for R-NaD on Google Colab, utilizing human play data downloaded from Discord.

In [ ]:
# @title Mount Google Drive and Install Basic Tools
from google.colab import drive
drive.mount('/content/drive')

!pip install uv -q
!uv pip install --system mlflow pyngrok -q

In [ ]:
# @title Clone Repository
import os

os.chdir("/content")

REPO_URL = "https://github.com/kitamura-tetsuo/R-NaD-StS2.git"
REPO_NAME = "R-NaD-StS2"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME}...")
    !git clone --recursive {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Pulling latest changes...")
    os.chdir(REPO_NAME)
    !git pull
    !git submodule update --init --recursive
    os.chdir("..")

os.chdir(REPO_NAME)
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# @title Setup Environment & Dependencies
import os
import subprocess

# 1. Install Rust (required for battle_simulator)
if not os.path.exists("/root/.cargo/bin/rustc"):
    print("Installing Rust...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] += ":/root/.cargo/bin"

# 2. Check GPU and install JAX
if 'CUDA_VERSION' in os.environ:
    print("GPU detected. Installing JAX for CUDA...")
    !nvidia-smi
    !uv pip install --system "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
else:
    print("GPU not detected. Offline training might be slow.")
    !uv pip install --system "jax[cpu]"

# 3. Install other Python dependencies from requirements.txt
print("Installing dependencies from R-NaD/requirements.txt...")
if os.path.exists("R-NaD/requirements.txt"):
    !uv pip install --system -r R-NaD/requirements.txt
else:
    !uv pip install --system dm-haiku optax open_spiel mlflow orbax-checkpoint httpx python-dotenv

# 4. Build Native Extensions
print("Building native extensions...")

# Build battle_simulator (Rust)
if os.path.exists("battle_simulator"):
    print("Building battle_simulator...")
    !cd battle_simulator && cargo build --release
    !cp battle_simulator/target/release/libbattle_simulator.so R-NaD/battle_simulator.so
    print("battle_simulator.so built and copied to R-NaD/")

# Build libpython_fixer (C)
if os.path.exists("R-NaD/setup.py"):
    print("Building libpython_fixer...")
    !cd R-NaD && python3 setup.py build_ext --inplace
    print("libpython_fixer built.")

In [ ]:
# @title Configuration & Secrets
from google.colab import userdata
import os

# Setup Discord Bot Token
try:
    token = userdata.get("DISVORD_BOT_TOKEN")
    with open(".env", "w") as f:
        f.write(f"DISVORD_BOT_TOKEN={token}\n")
    print("Discord Bot Token configured in .env")
except userdata.SecretNotFoundError:
    print("WARNING: \"DISVORD_BOT_TOKEN\" not found in Colab Secrets.")
    print("Please add it to the Secrets tab (key icon) to enable human data downloading.")

# Setup Drive Paths
DRIVE_ROOT = "/content/drive/MyDrive/R-NaD-StS2_experiments"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
REPLAY_DIR = os.path.join(DRIVE_ROOT, "replays")
TRAJECTORY_DIR = os.path.join(DRIVE_ROOT, "trajectories")
MLRUNS_DIR = os.path.join(DRIVE_ROOT, "mlruns")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(REPLAY_DIR, exist_ok=True)
os.makedirs(TRAJECTORY_DIR, exist_ok=True)
os.makedirs(MLRUNS_DIR, exist_ok=True)

# Set environment variables
os.environ["MLFLOW_TRACKING_URI"] = f"file://{MLRUNS_DIR}"
os.environ["RNAD_REPLAY_DIR"] = REPLAY_DIR
os.environ["RNAD_TRAJECTORY_DIR"] = TRAJECTORY_DIR

print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
print(f"Human replays will be saved to: {REPLAY_DIR}")
print(f"Machine trajectories will be saved to: {TRAJECTORY_DIR}")
print(f"MLflow runs will be saved to: {MLRUNS_DIR}")

# Seed trajectories from repo if Drive folder is fresh
print("Syncing trajectories from repository to Drive (if new)...")
!cp -n R-NaD/trajectories/*.json {TRAJECTORY_DIR}/ 2>/dev/null || true


In [ ]:
# @title Run Offline Training
import sys
import os

# Add R-NaD directory to sys.path
sys.path.append(os.path.join(os.getcwd(), "R-NaD"))

# Run the offline training script
# Parameters:
# --epochs: Number of epochs to run
# --data_dir: Path to human replays (Drive)
# --trajectory_dir: Path to machine trajectories (Drive)
# --checkpoint_dir: Path to checkpoints (Drive)
# --jax_platform: gpu or cpu

REPLAY_DIR = os.environ.get("RNAD_REPLAY_DIR")
TRAJECTORY_DIR = os.environ.get("RNAD_TRAJECTORY_DIR")
CHECKPOINT_DIR = os.path.join("/content/drive/MyDrive/R-NaD-StS2_experiments", "checkpoints")

!python R-NaD/train_offline_sts2.py \
    --epochs 100000 \
    --data_dir "{REPLAY_DIR}" \
    --trajectory_dir "{TRAJECTORY_DIR}" \
    --checkpoint_dir "{CHECKPOINT_DIR}" \
    --jax_platform gpu
